# M15 lab solution: Advanced Indexing Strategies

**Brief.** Index redesign: apply parent-document retrieval and contextual chunk headers to the baseline and report the change in hit rate and answer quality.

Run with the default environment: `uv run jupyter lab`.

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from helpers import get_embeddings, get_experientiallabs_llm

llm = get_experientiallabs_llm()
embeddings = get_embeddings()
LLM_NAME = llm.model_name
EMBEDDING_NAME = getattr(embeddings, "model", None) or getattr(embeddings, "endpoint", None)
print("LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)

In [ ]:
# The inline "Part V system", reused by the later labs: the handbook split as lab_kit.baseline splits it
# (recursive, 500 characters, 50 overlap), a FAISS vector retriever over those chunks, and an answer
# prompt that cites numbered passages and is allowed to say "I don't know".
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [
    Document(page_content=c, metadata={"source": p["source"], "section": p["title"]})
    for p in handbook
    for c in splitter.split_text(p["text"])
]
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions from Larkspur Cloud customers using only the numbered handbook passages. "
               "Cite the passages you use as [1], [2]. If the passages do not contain the answer, "
               "reply exactly: I don't know."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata.get('source', '?')}) {d.page_content}" for i, d in enumerate(docs, 1))


def answer(question, docs, config=None):
    return (ANSWER_PROMPT | llm).invoke({"context": format_docs(docs), "question": question}, config=config).content


print(len(chunks), "chunks indexed")

In [ ]:
def texts(docs):
    return [d.page_content for d in docs]


def score(results, questions=None):
    """results: question id -> retrieved chunk texts in rank order; scored with lab_kit on the questions with evidence."""
    qs = questions or golden
    return {"hit_rate": round(hit_rate(results, qs), 3), "recall_at_k": round(recall_at_k(results, qs), 3),
            "mrr": round(mrr(results, qs), 3)}

In [ ]:
from pydantic import BaseModel, Field


class Grade(BaseModel):
    correct: bool = Field(description="True when the answer states every fact the reference answer gives for this question.")
    faithful: bool = Field(description="True when every claim in the answer is supported by the passages.")


JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You grade one answer from a customer-support RAG system. Judge it against the reference answer "
               "and the passages the system was given. Ignore citation markers such as [1]."),
    ("human", "Question: {question}\n\nReference answer: {reference}\n\nPassages:\n{context}\n\nAnswer to grade: {answer}"),
])
judge = JUDGE_PROMPT | llm.with_structured_output(Grade)


def grade(g, docs, reply):
    return judge.invoke({"question": g["question"], "reference": g["reference_answer"],
                         "context": format_docs(docs), "answer": reply})


def is_refusal(reply):
    return reply.strip().lower().startswith("i don't know")

## 1. Three indexes

- **Baseline**: the Part V retriever above, top 4 of the 500-character chunks.
- **Parent-document retrieval**: `ParentDocumentRetriever` embeds 150-character children and returns their 600-character parents. The search key is small and sharp; the payload is large enough to hold a whole answer.
- **Contextual chunk headers**: each baseline chunk is embedded with a header naming the document and its section; the model reads the same header and chunk.

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore

SOURCES = [Document(page_content=p["text"], metadata={"source": p["source"], "section": p["title"]}) for p in handbook]

from langchain_core.vectorstores import InMemoryVectorStore

child_store = InMemoryVectorStore(embeddings)
pdr = ParentDocumentRetriever(
    vectorstore=child_store,
    docstore=InMemoryStore(),
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=0),
    parent_splitter=RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0),
    search_kwargs={"k": 8},
)
pdr.add_documents(SOURCES)


def pdr_search(question, k=4):
    return pdr.invoke(question)[:k]


headed = [Document(page_content=f"Larkspur Cloud handbook > {d.metadata['section']}\n\n{d.page_content}", metadata=d.metadata)
          for d in chunks]
header_store = FAISS.from_documents(headed, embeddings)

INDEXES = {
    "baseline": lambda q: retriever.invoke(q),
    "parent-document": pdr_search,
    "contextual headers": lambda q: header_store.similarity_search(q, k=4),
}

## 2. Retrieval and answers

Hit rate, recall and MRR at 4 come from `lab_kit` on the 25 questions with evidence. The mean characters sent to the model per question show what each index costs in context. Answer quality: correctness and faithfulness from the LLM judge, plus refusals on the 5 out-of-corpus questions.

In [ ]:
rows = []
for name, search in INDEXES.items():
    hits = {g["id"]: search(g["question"]) for g in golden}
    metrics = score({qid: texts(ds) for qid, ds in hits.items()})
    graded, refused = [], 0
    for g in golden:
        reply = answer(g["question"], hits[g["id"]])
        if g["evidence_spans"]:
            graded.append(grade(g, hits[g["id"]], reply))
        else:
            refused += is_refusal(reply)
    rows.append({"index": name, **metrics,
                 "mean_context_chars": round(sum(len(d.page_content) for ds in hits.values() for d in ds) / len(hits)),
                 "correctness": round(sum(v.correct for v in graded) / len(graded), 3),
                 "faithfulness": round(sum(v.faithful for v in graded) / len(graded), 3),
                 "ooc_refusals": f"{refused}/5"})
    print(rows[-1])

## Results

In [ ]:
import pandas as pd

print("run:", time.strftime("%Y-%m-%d"), "| LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)
print(pd.DataFrame(rows).to_string(index=False))

## What this shows

- Parent-document retrieval separates the search key from the payload: compare its hit rate with its context size, since a larger payload makes a hit more likely and costs more tokens per question.
- Contextual headers change only what gets embedded. Where a chunk never names its subject, the header gives the embedding the missing words; where chunks already name it, expect little change.
- Read correctness against the hit rate: a retrieval gain that does not move answer quality was not on the questions that failed.